In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import re

In [4]:
df_deepdive = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_deepdive/final_09_03_2026.csv")
df_market = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_market/final_marrket.csv")
df_usecase = pd.read_csv("/home/binperdok/KLTN2026/Data/data_label_usecase/final_usecase.csv")

In [5]:
df_deepdive.head(5)

,title,content,label
0,PRX Part 3 — Đào tạo một Text-to-Image Model t...,Chào mừng trở lại 👋 Trong hai bài viết trước (...,DEEP DIVE
1,Mixture of Experts (MoEs) trong Transformers,"Trong vài năm qua, việc mở rộng quy mô các mô ...",DEEP DIVE
2,Huấn luyện các mô hình AI với Unsloth và Huggi...,Bạn sẽ cầnChạy JobCài đặt SkillClaude CodeCode...,DEEP DIVE
3,Transformers.js v4 Preview: Hiện đã có mặt trê...,Cải thiện Hiệu suất & Thời gian chạy\nTái cấu ...,DEEP DIVE
4,Mô hình Holo2 mới của Công ty H dẫn đầu trong ...,Hai tháng kể từ khi phát hành đợt đầu tiên của...,DEEP DIVE


In [6]:
df_market.head()

,title,content,label
0,CEO Anthropic lên tiếng sau mâu thuẫn với Lầu ...,"""Chúng tôi thực hiện quyền của mình theo Tu ch...",MARKET SIGNALS
1,Xe vá ổ gà trong hai phút,"Sử dụng công nghệ DuraPatcher, xe Cimline P5 d...",NOISE
2,Chatbot Anthropic đứng đầu bảng ứng dụng miễn ...,"TheoBusiness Insider, tính đến cuối giờ chiều ...",MARKET SIGNALS
3,OpenAI tìm kiếm người không chuyên nhưng 'có gu',"Theo Altman, một trong những con đường tốt nhấ...",MARKET SIGNALS
4,ChatGPT chạm mốc 900 triệu người dùng,Thông báo mới cho thấy ChatGPT đang tiến gần h...,MARKET SIGNALS


In [7]:
df_usecase.head()

,title,content,label
0,Ứng Dụng AI Trong Y Tế: Cách Mạng Hóa Chăm Sóc...,Bạn có tin rằng AI hiện nay AI đã chẩn đoán un...,SOLUTIONS & USE CASES
1,Xu Hướng Ứng Dụng AI Trong Du Lịch: 20+ Cách T...,Ngành du lịch đang đối mặt với một thách thức ...,SOLUTIONS & USE CASES
2,Ứng dụng AI cho từng phòng ban: Giải pháp tối ...,"Theo tạp chí Forbes, có tới 83% công ty tuyên ...",SOLUTIONS & USE CASES
3,5 Ứng dụng AI trong kinh doanh giúp doanh nghi...,Trí tuệ nhân tạo (AI) đã không còn là công ngh...,SOLUTIONS & USE CASES
4,"Ứng dụng AI trong Bất động sản 2025: Xu hướng,...",Shark Hưng từng nhận định: “Mô hình môi giới B...,SOLUTIONS & USE CASES


In [8]:
data = pd.concat([df_usecase, df_deepdive , df_market], axis = 0, ignore_index=True)
data.head()

,title,content,label
0,Ứng Dụng AI Trong Y Tế: Cách Mạng Hóa Chăm Sóc...,Bạn có tin rằng AI hiện nay AI đã chẩn đoán un...,SOLUTIONS & USE CASES
1,Xu Hướng Ứng Dụng AI Trong Du Lịch: 20+ Cách T...,Ngành du lịch đang đối mặt với một thách thức ...,SOLUTIONS & USE CASES
2,Ứng dụng AI cho từng phòng ban: Giải pháp tối ...,"Theo tạp chí Forbes, có tới 83% công ty tuyên ...",SOLUTIONS & USE CASES
3,5 Ứng dụng AI trong kinh doanh giúp doanh nghi...,Trí tuệ nhân tạo (AI) đã không còn là công ngh...,SOLUTIONS & USE CASES
4,"Ứng dụng AI trong Bất động sản 2025: Xu hướng,...",Shark Hưng từng nhận định: “Mô hình môi giới B...,SOLUTIONS & USE CASES


In [9]:
data.label.value_counts()

label
MARKET SIGNALS           593
NOISE                    275
DEEP DIVE                257
SOLUTIONS & USE CASES    231
Name: count, dtype: int64

In [10]:
data.isnull().sum()

title       0
content     0
label      42
dtype: int64

In [11]:
data = data.dropna(subset = ['label'])
data.isnull().sum()

title      0
content    0
label      0
dtype: int64

In [12]:
(data == "").any().any()   

np.False_

In [13]:
data.head()

,title,content,label
0,Ứng Dụng AI Trong Y Tế: Cách Mạng Hóa Chăm Sóc...,Bạn có tin rằng AI hiện nay AI đã chẩn đoán un...,SOLUTIONS & USE CASES
1,Xu Hướng Ứng Dụng AI Trong Du Lịch: 20+ Cách T...,Ngành du lịch đang đối mặt với một thách thức ...,SOLUTIONS & USE CASES
2,Ứng dụng AI cho từng phòng ban: Giải pháp tối ...,"Theo tạp chí Forbes, có tới 83% công ty tuyên ...",SOLUTIONS & USE CASES
3,5 Ứng dụng AI trong kinh doanh giúp doanh nghi...,Trí tuệ nhân tạo (AI) đã không còn là công ngh...,SOLUTIONS & USE CASES
4,"Ứng dụng AI trong Bất động sản 2025: Xu hướng,...",Shark Hưng từng nhận định: “Mô hình môi giới B...,SOLUTIONS & USE CASES


In [14]:
# Trộn toàn bộ dữ liệu, giữ nguyên số dòng
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
target_n = 275   # hoặc 250, 300... tùy bạn

# Tách ra 2 phần: MARKET SIGNALS và phần còn lại
market = data[data['label'] == 'MARKET SIGNALS']
others = data[data['label'] != 'MARKET SIGNALS']

# Lấy ngẫu nhiên target_n dòng từ MARKET SIGNALS
market_down = market.sample(n=target_n, random_state=42)

# Gộp lại
data_balanced = pd.concat([market_down, others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

data_balanced.label.value_counts()

label
MARKET SIGNALS           275
NOISE                    275
DEEP DIVE                257
SOLUTIONS & USE CASES    231
Name: count, dtype: int64

In [16]:
data_balanced.to_csv("/home/binperdok/KLTN2026/Data/FINAL_DATA.csv")

In [17]:
data_balanced.label.value_counts()['title'] = data_balanced['title'].fillna('')
data_balanced['content'] = data_balanced['content'].fillna('')

data_balanced['text'] = data_balanced['title'] + ' ' + data_balanced['content']

In [18]:
# xử lý đưa về chữ thường
data_balanced['text'] = data_balanced['text'].str.lower()

In [19]:
# xoa url, email, html tag
def remove_urls_emails_html(x):
    x = re.sub(r'http\S+|www\.\S+', ' ', x)        # URL
    x = re.sub(r'\S+@\S+\.\S+', ' ', x)           # email
    x = re.sub(r'<.*?>', ' ', x)                  # HTML
    return x

data_balanced['text'] = data_balanced['text'].apply(remove_urls_emails_html)

In [20]:
# Xoa ki tu dac biet
def remove_special_chars(x):
    x = re.sub(r'[^a-zA-ZÀ-ỹà-ỹ\s]', ' ', x)  # nếu có tiếng Việt
    return x

data_balanced['text'] = data_balanced['text'].apply(remove_special_chars)

In [21]:
#Xoa khong thua o dau va cuoi 
data_balanced['text'] = data_balanced['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [22]:
from underthesea import word_tokenize

def tokenize_text(s):
    s = str(s)
    return word_tokenize(s, format="text")

data_balanced['text_tok'] = data_balanced['text'].apply(tokenize_text)

In [23]:
data_balanced.head() 

,title,content,label,text,text_tok
0,Ứng dụng AI trong y tế cần phải song hành cùng...,Nội dung trên được các chuyên gia đưa ra thảo ...,MARKET SIGNALS,ứng dụng ai trong y tế cần phải song hành cùng...,ứng_dụng ai trong y_tế cần phải song_hành cùng...
1,Khai mạc khóa tập huấn “Ứng dụng AI trong kỹ n...,"Sáng 13-11, tại phường Nha Trang, Trung tâm Bồ...",SOLUTIONS & USE CASES,khai mạc khóa tập huấn ứng dụng ai trong kỹ nă...,khai_mạc khóa tập_huấn ứng_dụng ai trong kỹ_nă...
2,"Chàng trai làm game chỉ bằng 1 câu lệnh AI, ki...","Được tạo ra bằng AI, Fly.pieter.com là trò chơ...",SOLUTIONS & USE CASES,chàng trai làm game chỉ bằng câu lệnh ai kiếm ...,chàng trai làm game chỉ bằng câu lệnh ai kiếm ...
3,Từ hôm nay 1/3: Luật trí tuệ nhân tạo có hiệu ...,"Từ hôm nay 01/03/2026, Luật Trí tuệ nhân tạo s...",MARKET SIGNALS,từ hôm nay luật trí tuệ nhân tạo có hiệu lực n...,từ hôm_nay luật_trí_tuệ nhân_tạo có hiệu_lực n...
4,Mondelez Kinh Đô đón đầu công nghệ AI để thúc ...,Các sản phẩm của Mondelez Kinh Đô hiện đã bao ...,SOLUTIONS & USE CASES,mondelez kinh đô đón đầu công nghệ ai để thúc ...,mondelez kinh_đô đón_đầu công_nghệ ai để thúc_...


In [24]:
import unicodedata2

def normalize_unicode(text):
    return unicodedata2.normalize('NFC', str(text))

data_balanced['text_tok'] = data_balanced['text_tok'].apply(normalize_unicode)

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data_balanced['label_enc'] = le.fit_transform(data_balanced['label'])

print(dict(zip(le.classes_, le.transform(le.classes_))))
# Ví dụ: {'DEEP DIVE': 0, 'MARKET SIGNALS': 1, 'NOISE': 2, 'SOLUTIONS & USE CASES': 3}

{'DEEP DIVE': np.int64(0), 'MARKET SIGNALS': np.int64(1), 'NOISE': np.int64(2), 'SOLUTIONS & USE CASES': np.int64(3)}


In [26]:
data_balanced[['text_tok', 'label', 'label_enc']].to_csv(
    "/home/binperdok/KLTN2026/Data/PROCESSED_DATA.csv", index=False
)